# Практическая работа 3. Чтение журнала событий по частям

Вариант 8

В файле `events_lines.json` одна строка соответствует одному событию. Для этой работы используем только этот журнал. Условие активности — строго больше десяти событий.

In [1]:
from collections import Counter
from pathlib import Path

import pandas as pd
from shop_data import DATA, read_orders

out = Path("results")
out.mkdir(exist_ok=True)

### 1–2. Чтение блоками и подсчёт событий

Читаем по 250 строк. Один счётчик накапливает итоги по типам событий, второй — по пользователям.

In [2]:
by_type = Counter()
by_user = Counter()
for block in pd.read_json(DATA / "events_lines.json", lines=True, chunksize=250):
    by_type.update(block["event_type"].value_counts().to_dict())
    by_user.update(block["user_id"].value_counts().to_dict())

print("Обработано событий:", sum(by_type.values()))

Обработано событий: 2000


### 3. Сводка по типам событий

In [3]:
event_counts = (
    pd.Series(by_type, name="events_count")
    .rename_axis("event_type")
    .sort_values(ascending=False)
    .reset_index()
)
event_counts.to_csv(out / "event_counts.csv", index=False)
event_counts

,event_type,events_count
0,click,416
1,purchase,413
2,scroll,405
3,add_to_cart,387
4,page_view,379


### 4. Заказы активных пользователей

К каждому заказу присоединяем ровно один итог по пользователю. Несколько событий не должны размножать строки заказов.

In [4]:
user_counts = pd.Series(by_user, name="events_count").rename_axis("user_id")
active = user_counts[user_counts > 10].sort_index()
orders = read_orders()
active_orders = orders.merge(active, on="user_id", how="inner", validate="many_to_one")
print("Заказов активных пользователей:", len(active_orders))
active_orders[["order_id", "user_id", "order_datetime", "events_count"]].head()

Заказов активных пользователей: 15000


,order_id,user_id,order_datetime,events_count
0,1,1044,2024-01-03 18:39:26,20
1,2,1079,2024-01-03 17:45:43,16
2,3,1041,2024-01-06 22:46:38,21
3,4,1083,2024-01-08 21:03:51,19
4,5,1068,2024-01-03 23:26:53,17


### 5. Список идентификаторов

В список входят все активные пользователи журнала, в том числе если у них нет заказов.

In [5]:
active.index.to_frame(index=False).to_csv(out / "active_users.txt", header=False, index=False)
print("Активных пользователей:", len(active))
print("Первые десять ID:", active.index[:10].tolist())

Активных пользователей: 100
Первые десять ID: [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010]
